# 04 — Eksperimen Pendukung
Tiga eksperimen yang dipakai pada Bab 4.3–4.4 laporan:
1. **Fitur pretrained tanpa fine-tuning** — pendekatan referensi utama kedua (Prakash & Singh, 2025), dijalankan pada
   data dan protokol yang sama (dihitung langsung di notebook ini).
2. **Uji kebocoran data** — membandingkan pembagian per pasien dengan pembagian acak per citra.
3. **Pengulangan tiga seed** — rata-rata ± simpangan baku dan pemilihan *proposed model* berdasarkan data validasi.

Pelatihan ulang memakan waktu lama sehingga memakai saklar `JALANKAN_PELATIHAN` (bawaan `False`);
tanpa pelatihan ulang, notebook membaca hasil pelatihan tersimpan di `outputs/`.

In [1]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

PROJECT = Path.cwd().parent                      # folder "Tugas Temu Kembali Citra"
RAW_DIR = PROJECT / "data pneumonia" / "chest_xray"
DATA_DIR = PROJECT / "data"                      # split dan cache citra hasil persiapan data
OUTPUTS = PROJECT / "outputs"                    # model dan hasil eksperimen
HASIL = Path("hasil")                            # keluaran notebook (terpisah dari outputs/)
HASIL.mkdir(exist_ok=True)
CLASSES = ["Normal", "Bacteria", "Virus"]
NAMA_KELAS = {"Normal": "Normal", "Bacteria": "Pneumonia Bakteri", "Virus": "Pneumonia Virus"}
print("Folder project:", PROJECT)

Folder project: C:\Users\LENOVO\Documents\Temu Kembali Citra\Tugas\Tugas Temu Kembali Citra


In [2]:
import json
import numpy as np
import pandas as pd
import torch, torch.nn as nn
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.model_selection import train_test_split

JALANKAN_PELATIHAN = False
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODELS = ["mobilenet_v2", "efficientnet_b0", "densenet121"]
MODEL_LABELS = {"mobilenet_v2": "MobileNetV2", "efficientnet_b0": "EfficientNet-B0", "densenet121": "DenseNet121"}
SCENARIO_LABELS = {"s1_original": "Skenario 1 (Original)", "s2_augment": "Skenario 2 (Augmentasi)",
                   "s3_clahe_augment": "Skenario 3 (CLAHE + Augmentasi)"}
TF = transforms.Compose([transforms.Resize((224, 224)), transforms.Grayscale(3), transforms.ToTensor(),
                         transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
splits = pd.read_csv(DATA_DIR / "splits.csv")

## 1. Fitur pretrained ImageNet tanpa fine-tuning
Bobot ImageNet dipakai apa adanya (seperti referensi utama kedua), lalu temu kembali dengan *cosine similarity*
pada galeri dan kueri yang sama dengan model *fine-tuned*.

In [3]:
class XrayDataset(Dataset):
    def __init__(self, df):
        self.paths = [DATA_DIR / "cache" / "original" / n for n in df["cache_name"]]
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        return TF(Image.open(self.paths[i]).convert("L"))

def pretrained_backbone(arch):
    pool = [nn.AdaptiveAvgPool2d(1), nn.Flatten()]
    if arch == "mobilenet_v2":
        return nn.Sequential(models.mobilenet_v2(weights="IMAGENET1K_V1").features, *pool)
    if arch == "efficientnet_b0":
        return nn.Sequential(models.efficientnet_b0(weights="IMAGENET1K_V1").features, *pool)
    return nn.Sequential(models.densenet121(weights="IMAGENET1K_V1").features, nn.ReLU(), *pool)

@torch.no_grad()
def embed(net, df):
    out = []
    for x in DataLoader(XrayDataset(df), batch_size=64):
        with torch.autocast("cuda", dtype=torch.float16, enabled=DEVICE.type == "cuda"):
            out.append(net(x.to(DEVICE)).float().cpu())
    return torch.cat(out).numpy()

def retrieval_metrics(q, ql, g, gl):
    q = q / np.linalg.norm(q, axis=1, keepdims=True); g = g / np.linalg.norm(g, axis=1, keepdims=True)
    rel = (gl[np.argsort(-(q @ g.T), axis=1)] == ql[:, None]).astype(np.float32)
    ap = (rel * np.cumsum(rel, 1) / np.arange(1, rel.shape[1] + 1)).sum(1) / rel.sum(1)
    return {"P@10": float(rel[:, :10].mean() * 100), "mAP": float(ap.mean() * 100),
            "mAP Virus": float(ap[ql == 2].mean() * 100)}

train = splits[splits.split == "train"].reset_index(drop=True)
test = splits[splits.split == "test"].reset_index(drop=True)
rows = []
for a in MODELS:
    net = pretrained_backbone(a).to(DEVICE).eval()
    m = retrieval_metrics(embed(net, test), test["label_idx"].to_numpy(), embed(net, train), train["label_idx"].to_numpy())
    rows.append({"Model": MODEL_LABELS[a], **m})
pretrained = pd.DataFrame(rows).set_index("Model").round(1)
pretrained

,P@10,mAP,mAP Virus
Model,,,
MobileNetV2,70.0,54.1,34.8
EfficientNet-B0,67.5,51.9,32.6
DenseNet121,68.0,52.5,33.8


In [4]:
seed_runs = pd.read_csv(OUTPUTS / "extra" / "seed_runs.csv")
s1 = seed_runs[seed_runs["Skenario"] == SCENARIO_LABELS["s1_original"]].groupby("Model")[["P@10", "mAP"]].mean() * 100
banding = pretrained[["P@10", "mAP"]].join(s1.round(1), lsuffix=" pretrained", rsuffix=" fine-tuned (rata-rata 3 seed)")
banding["Kenaikan mAP (poin)"] = (banding["mAP fine-tuned (rata-rata 3 seed)"] - banding["mAP pretrained"]).round(1)
banding

,P@10 pretrained,mAP pretrained,P@10 fine-tuned (rata-rata 3 seed),mAP fine-tuned (rata-rata 3 seed),Kenaikan mAP (poin)
Model,,,,,
MobileNetV2,70.0,54.1,80.2,80.1,26.0
EfficientNet-B0,67.5,51.9,77.7,77.0,25.1
DenseNet121,68.0,52.5,79.6,81.2,28.7


## 2. Uji kebocoran data: pembagian acak per citra
Data yang sama dibagi acak per citra (70/15/15) tanpa memperhatikan pasien, sehingga citra dari pasien yang sama
dapat muncul di data latih dan data uji.

In [5]:
acak = splits.copy()
idx = np.arange(len(acak))
tr_i, rest = train_test_split(idx, test_size=0.30, stratify=acak["label"], random_state=42)
va_i, te_i = train_test_split(rest, test_size=0.50, stratify=acak["label"].iloc[rest], random_state=42)
acak["split"] = "train"; acak.loc[va_i, "split"] = "val"; acak.loc[te_i, "split"] = "test"
acak.to_csv(HASIL / "splits_random_image.csv", index=False)

pasien_latih = set(acak.loc[acak.split == "train", "patient"])
uji = acak[acak.split == "test"]
bocor = uji["patient"].isin(pasien_latih)
print(f"Citra uji yang pasiennya juga ada di data latih: {bocor.sum()} dari {len(uji)} ({bocor.mean()*100:.1f}%)")
print("Pada pembagian per pasien (penelitian ini): 0 citra")

Citra uji yang pasiennya juga ada di data latih: 523 dari 874 (59.8%)
Pada pembagian per pasien (penelitian ini): 0 citra


In [6]:
# Pelatihan model pada split acak memakai fungsi train_model di notebook 02 (Skenario 2, seed 42).
if JALANKAN_PELATIHAN:
    print("Jalankan train_model('s2_augment', arch) dari notebook 02 dengan splits_random_image.csv")
leak = pd.read_csv(OUTPUTS / "extra" / "leakage_comparison.csv")
tabel = leak.set_index(["Model", "Pembagian data"]).mul(100).round(1)
display(tabel)
for m in leak["Model"].unique():
    sub = leak[leak.Model == m].set_index("Pembagian data")["Macro F1"] * 100
    print(f"{m}: macro F1 naik {sub['Acak per citra'] - sub['Per pasien (penelitian ini)']:.1f} poin karena kebocoran")

Akurasi  Macro F1  Macro AUC  \
Model           Pembagian data                                              
MobileNetV2     Per pasien (penelitian ini)     80.6      80.7       91.4   
                Acak per citra                  83.5      82.4       94.0   
EfficientNet-B0 Per pasien (penelitian ini)     81.4      81.9       93.0   
                Acak per citra                  82.8      82.0       94.5   

                                             P@10   mAP  
Model           Pembagian data                           
MobileNetV2     Per pasien (penelitian ini)  80.3  79.0  
                Acak per citra               81.0  77.4  
EfficientNet-B0 Per pasien (penelitian ini)  81.8  80.8  
                Acak per citra               81.1  79.0

MobileNetV2: macro F1 naik 1.7 poin karena kebocoran
EfficientNet-B0: macro F1 naik 0.2 poin karena kebocoran


## 3. Pengulangan tiga seed dan pemilihan proposed model
Setiap konfigurasi dilatih ulang dengan seed 7 dan 123 (selain 42). *Proposed model* adalah konfigurasi dengan
rata-rata **macro F1 validasi** tertinggi; checkpoint yang dipakai aplikasi adalah seed dengan macro F1 validasi
tertinggi pada konfigurasi tersebut. Data uji tidak dipakai untuk memilih model.

In [7]:
if JALANKAN_PELATIHAN:
    print("Jalankan train_model(skenario, model, seed=s) dari notebook 02 untuk seed 7 dan 123")
agg = seed_runs.groupby(["Skenario", "Model"], sort=False)[["Val Macro F1", "Macro F1", "P@10", "mAP"]].agg(["mean", "std"])
tampil = pd.DataFrame(index=agg.index)
for k in ["Val Macro F1", "Macro F1", "P@10", "mAP"]:
    tampil[k] = [f"{m*100:.1f} ± {s*100:.1f}" for m, s in zip(agg[(k, "mean")], agg[(k, "std")])]
display(tampil)

terbaik = agg[("Val Macro F1", "mean")].idxmax()
kandidat = seed_runs[(seed_runs.Skenario == terbaik[0]) & (seed_runs.Model == terbaik[1])]
seed_terbaik = int(kandidat.loc[kandidat["Val Macro F1"].idxmax(), "seed"])
print("Proposed model:", terbaik, "| checkpoint seed:", seed_terbaik)
print("Tercatat di outputs/best_run.json:", json.loads((OUTPUTS / "best_run.json").read_text()))

Val Macro F1    Macro F1  \
Skenario                        Model                                      
Skenario 1 (Original)           MobileNetV2       83.0 ± 0.4  80.4 ± 0.4   
                                EfficientNet-B0   82.7 ± 0.4  79.4 ± 1.8   
                                DenseNet121       82.8 ± 0.7  79.9 ± 1.1   
Skenario 2 (Augmentasi)         MobileNetV2       82.8 ± 0.3  80.3 ± 0.8   
                                EfficientNet-B0   83.4 ± 0.7  80.1 ± 2.0   
                                DenseNet121       84.0 ± 0.1  81.2 ± 0.9   
Skenario 3 (CLAHE + Augmentasi) MobileNetV2       82.0 ± 0.6  80.2 ± 1.3   
                                EfficientNet-B0   83.7 ± 0.7  80.9 ± 0.1   
                                DenseNet121       83.8 ± 1.0  79.4 ± 1.6   

                                                       P@10         mAP  
Skenario                        Model                                    
Skenario 1 (Original)           MobileNetV2      80.2 ± 1.8  80.1 ± 5.2  
                                EfficientNet-B0  77.7 ± 2.2  77.0 ± 6.6  
                                DenseNet121      79.6 ± 1.2  81.2 ± 4.0  
Skenario 2 (Augmentasi)         MobileNetV2      79.2 ± 2.1  77.0 ± 3.6  
                                EfficientNet-B0  80.1 ± 2.6  78.1 ± 4.1  
                                DenseNet121      80.3 ± 0.4  81.7 ± 2.1  
Skenario 3 (CLAHE + Augmentasi) MobileNetV2      78.3 ± 1.7  76.1 ± 4.0  
                                EfficientNet-B0  80.5 ± 0.9  79.8 ± 2.0  
                                DenseNet121      78.7 ± 1.7  79.6 ± 4.2

Proposed model: ('Skenario 2 (Augmentasi)', 'DenseNet121') | checkpoint seed: 7
Tercatat di outputs/best_run.json: {'scenario': 's2_augment', 'model': 'densenet121', 'tag': 'seed7', 'seed': 7}


## 4. Perbandingan dengan referensi utama (Saranya & Chinnadurai, 2025) pada tugas normal vs pneumonia
Referensi utama hanya membedakan dua kelas. Prediksi tiga kelas digabung (bakteri + virus = pneumonia) agar setara.
Kode lengkap: `src/binary_comparison.py`. Nilai referensi: akurasi 98,60%, presisi 98,49%, recall 97,40%, F1 98,50%.

In [8]:
biner = pd.read_csv(OUTPUTS / "extra" / "binary_comparison.csv").set_index("Model") * 100
referensi = pd.DataFrame([{"Akurasi": 98.60, "Presisi": 98.49, "Recall": 97.40, "F1": 98.50}],
                         index=["Saranya & Chinnadurai 2025 (dilaporkan)"])
pd.concat([referensi, biner[["Akurasi", "Presisi", "Recall", "F1", "P@10", "P@10 + RF"]]]).round(2)

,Akurasi,Presisi,Recall,F1,P@10,P@10 + RF
Saranya & Chinnadurai 2025 (dilaporkan),98.60,98.49,97.40,98.50,NaN,NaN
"Proposed (DenseNet121 S2), split per pasien",97.27,98.66,97.65,98.15,97.31,98.64
"MobileNetV2 S2, split per pasien",97.38,98.96,97.50,98.22,97.13,98.31
"MobileNetV2 S2, split acak per citra",96.45,99.19,95.92,97.53,96.59,98.16
"EfficientNet-B0 S2, split per pasien",97.16,99.40,96.76,98.06,97.09,98.78
"EfficientNet-B0 S2, split acak per citra",96.45,98.71,96.39,97.54,96.29,98.14


## 5. Reproduksi protokol Benmalek dkk. (2025)
Protokol dari naskah lengkap: folder latih Kaggle, split acak per citra 80/20, input 227×227, Adam lr 1e-4,
tanpa bobot kelas dan augmentasi. Kode lengkap: `src/reproduce_benmalek.py`. Nilai dilaporkan: akurasi 95,8%, F1 95,7%.

In [9]:
bm = pd.read_csv(OUTPUTS / "extra" / "benmalek_protocol.csv").set_index("Model") * 100
dilaporkan = pd.DataFrame([{"Akurasi": 95.8, "Weighted F1": 95.7, "F1 Normal": 99.6, "F1 Bakteri": 95.8,
                            "F1 Virus": 91.7}], index=["MobileNet Benmalek dkk. (dilaporkan)"])
pd.concat([dilaporkan, bm[["Akurasi", "Weighted F1", "F1 Normal", "F1 Bakteri", "F1 Virus"]]]).round(2)

,Akurasi,Weighted F1,F1 Normal,F1 Bakteri,F1 Virus
MobileNet Benmalek dkk. (dilaporkan),95.80,95.70,99.60,95.80,91.70
"MobileNetV2, reproduksi protokol Benmalek",83.43,83.06,97.03,84.64,66.13
DenseNet121 + augmentasi (metode penelitian ini),83.62,83.16,96.20,85.21,66.26


## 6. Relevance feedback pada proposed model (rata-rata ± simpangan baku tiga seed)

In [10]:
rf = pd.read_csv(OUTPUTS / "extra" / "relevance_feedback_summary.csv", index_col=[0, 1]) * 100
rf.round(2)

P@10 mean  P@10 std  mAP mean  mAP std  \
Fitur      Feedback                                                    
Fine-tuned Tanpa feedback         80.29      0.38     81.74     2.06   
           Rocchio 1 putaran      85.43      0.62     84.24     2.31   
           Rocchio 2 putaran      88.11      0.80     85.71     2.56   
Pretrained Tanpa feedback         67.96       NaN     52.50      NaN   
           Rocchio 1 putaran      79.39       NaN     54.45      NaN   
           Rocchio 2 putaran      84.87       NaN     55.46      NaN   

                              mAP Virus mean  mAP Virus std  
Fitur      Feedback                                          
Fine-tuned Tanpa feedback              63.69           5.00  
           Rocchio 1 putaran           68.10           5.53  
           Rocchio 2 putaran           70.90           5.94  
Pretrained Tanpa feedback              33.79            NaN  
           Rocchio 1 putaran           35.33            NaN  
           Rocchio 2 putaran           36.36            NaN